# 01 baseline

Step 1-3 - infrastructure, the frozen split, and plain steering.

Gate: B0/B1 must reproduce the old repo's baseline before anything is trained.

## Шаг 0. Настройка

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import torch

from steering import io

print(f"python  {sys.version.split()[0]}")
print(f"torch   {torch.__version__}")
print(f"mps     {torch.backends.mps.is_available()}")
print(f"repo    {io.REPO_ROOT}")

python  3.12.0
torch   2.13.0
mps     True
repo    /Users/bg/Coding/steering-mvp


### Проверка модуля для управления сохраненными результатами (scr/io.py)

In [2]:
# First call computes; second loads. Then a changed config must refuse to reuse the stale file.
cfg = {"model": "gpt2", "layer": 6, "seed": 0, "version": 1}

io.run_or_load("_smoke", cfg, lambda: {"hello": "world"})
io.run_or_load("_smoke", cfg, lambda: {"hello": "world"})

try:
    io.run_or_load("_smoke", {**cfg, "seed": 1}, lambda: {"hello": "other"})
except io.CacheMismatch as e:
    print(f"\nrefused, as it should:\n{e}")

computed _smoke.json  (7d44b47fe7adc931)
cached  _smoke.json  (7d44b47fe7adc931)

refused, as it should:
_smoke.json was produced by a different config.
  seed: 0 -> 1

Pass force=True to recompute and overwrite, or rename the run.


In [3]:
for f in io.RESULTS.glob("_smoke.*"):
    f.unlink()

### Загрузка модели

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

from steering import hooks

MODEL, LAYER = "gpt2", 6

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL).eval()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [3]:
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained("gpt2").eval().to(DEVICE)
print(f"device: {DEVICE}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

device: mps


In [4]:
print(f"{MODEL}: {hooks.n_layers(model)} blocks, hook at layer {LAYER} "
      f"(= hidden_states[{hooks.hidden_state_index(LAYER)}])")

gpt2: 12 blocks, hook at layer 6 (= hidden_states[7])


In [5]:
batch = tokenizer(["The capital of France is"], return_tensors="pt").to(DEVICE)

with hooks.ResidualHook(model, layer=LAYER, capture=True) as hook, torch.no_grad():
    model(**batch)

h = hook.captured[0][0]                     # [T, D]
print("norm per position:", [f"{v:.0f}" for v in h.norm(dim=-1)])
print("top outlier dims: ", h.abs().max(0).values.topk(3).indices.tolist())

norm per position: ['3046', '92', '89', '105', '85']
top outlier dims:  [447, 138, 378]


In [6]:
with hooks.ResidualHook(model, layer=LAYER) as hook:
    model.generate(**batch, max_new_tokens=8, do_sample=False,
                   pad_token_id=tokenizer.eos_token_id)

print(f"seq_lens        {hook.seq_lens}")
print(f"prefill         {hook.prefill_len} tokens")
print(f"decode steps    {hook.n_decode_steps}")

seq_lens        [5, 1, 1, 1, 1, 1, 1, 1]
prefill         5 tokens
decode steps    7


In [7]:
from steering import spaces

print("gpt2:      ", spaces.should_center("gpt2"))
print("gemma-2-it:", spaces.should_center("google/gemma-2-2b-it"))
try:
    spaces.should_center("some-model-nobody-measured")
except ValueError as e:
    print(f"\nrefused: {e}")

gpt2:       True
gemma-2-it: False

refused: unknown architecture 'some-model-nobody-measured': whether centering is output-neutral has not been measured for it. Measure the relative logit change under centering (see DEVLOG 2026-08-22) and add it to spaces._CENTERS before using this model.


Проверяем как работает hook

In [8]:
with hooks.ResidualHook(model, layer=LAYER, capture=True) as capture_hook, torch.no_grad():
    model(**batch)

h = capture_hook.captured[0]
scale = spaces.activation_scale(h, exclude_sink=True)
print(f"E||h|| (median, sink excluded): {scale:.1f}")

z = spaces.encode(h, scale=scale, center=True)
back = spaces.decode(z, scale=scale, center=True)
print("decode(encode(h)) == h:        ", torch.allclose(back, h))
print("decode(encode(h)) == center(h):", torch.allclose(back, spaces.center(h)))

E||h|| (median, sink excluded): 88.5
decode(encode(h)) == h:         False
decode(encode(h)) == center(h): True


### Загрузка SAE

In [9]:
from steering import vectors

sae = vectors.load_sae("gpt2-small-resid-post-v5-128k", "blocks.6.hook_resid_post",
                        layer=LAYER, d_model=768, device=DEVICE)
print(f"d_sae={sae.cfg.d_sae}  k={sae.cfg.k}  hook_name={sae.cfg.metadata.hook_name}")

try:
    vectors.load_sae("gpt2-small-resid-post-v5-128k", "blocks.6.hook_resid_post",
                      layer=7, d_model=768, device=DEVICE)
except ValueError as e:
    print(f"\nrefused: {e}")

d_sae=131072  k=32  hook_name=blocks.6.hook_resid_post

refused: SAE hook_name 'blocks.6.hook_resid_post' does not match intervention layer 7. Expected one of ['blocks.7.hook_resid_post', 'blocks.8.hook_resid_pre']. Remember resid_post(L) == resid_pre(L+1).


In [10]:
import glob
import pandas as pd

pq = glob.glob(str(io.REPO_ROOT / "../steering-final/**/pile-10k*/**/*.parquet"), recursive=True)
# or point this at wherever your Pile-10k parquet lives; any few hundred short texts will do
texts = pd.read_parquet(pq[0])["text"].head(200).tolist() if pq else None

if texts:
    center = spaces.should_center("gpt2")
    stats = vectors.compute_feature_stats(model, tokenizer, sae, layer=LAYER, texts=texts,
                                          batch_size=8, max_length=64, center=center)
    print(f"mean_l0 = {float(stats['mean_l0']):.2f}  (k = {sae.cfg.k})")
    lo, hi = vectors.band_from_mean(stats, lo=0.41, hi=20.5)
    print(f"band: [{lo:.2e}, {hi:.2e}]  -> {vectors.frequency_band(stats, lo, hi).numel()} in-band")

In [11]:
import torch as _torch

fake_stats = {"frequency": _torch.zeros(2000), "n_tokens": _torch.tensor(50_000)}
fake_stats["frequency"][200:800] = 2.44e-4  # 600 in-band candidates

split = vectors.select_and_freeze_split(
    fake_stats, io.RESULTS / "_smoke_split.json",
    freq_min=1e-4, freq_max=5e-3, n_dev=20, n_test=50, seed=0,
    model="gpt2", source="_smoke",
)
print(f"dev={len(split.dev)} test={len(split.test)} train_pool={len(split.train_pool())}")
print(f"fingerprint {split.fingerprint()}")

import json
p = io.RESULTS / "_smoke_split.json"
payload = json.loads(p.read_text())
payload["test"][0] = payload["test"][0] + 1  # tamper
p.write_text(json.dumps(payload))
try:
    vectors.load_split(p)
except ValueError as e:
    print(f"\nrefused: {e}")

for f in io.RESULTS.glob("_smoke_split.json"):
    f.unlink()

dev=20 test=50 train_pool=1930
fingerprint 7cc2a93c9fcda421

refused: HOLDOUT VIOLATION: split at /Users/bg/Coding/steering-mvp/results/_smoke_split.json does not match its fingerprint (be2cf35c2c422e1d != 7cc2a93c9fcda421). The frozen file was edited.


## Шаг 1. Проверка обычного steering'а

In [12]:
import json

FEATURE_IDS = [1878, 12789, 13836, 15452, 31544, 32801, 37444, 46144,
               53354, 60559, 60695, 65056, 67922, 74485, 89756, 130710]
prompts = json.loads((io.REPO_ROOT / "configs" / "prompts_neutral_32.json").read_text())["prompts"]
directions = vectors.steering_directions(sae, FEATURE_IDS)  # unit-norm, [16, 768]
print(len(prompts), "prompts,", directions.shape[0], "features")

32 prompts, 16 features


In [ ]:
from steering import generate, interventions, metrics
import statistics

R_GRID = [0.0, 0.2, 0.4, 0.6, 0.8, 1]
scale = spaces.activation_scale(capture_hook.captured[0], exclude_sink=True)

b0_out = generate.generate(model, tokenizer, prompts, interventions.NoSteering(),
                           layer=LAYER, device=DEVICE, max_new_tokens=32, batch_size=8, seed=0)
b0_nll = metrics.reference_nll(model, tokenizer, prompts, b0_out.texts, device=DEVICE)
b0_ppl = float(b0_nll[~b0_nll.isnan()].mean().exp())
print(f"B0 ppl = {b0_ppl:.2f}")

rows = []
for r in R_GRID:
    for fid, v in zip(FEATURE_IDS, directions):
        iv = interventions.NoSteering() if r == 0.0 else interventions.AdditiveSteering(v, r, scale)
        out = generate.generate(model, tokenizer, prompts, iv, layer=LAYER, device=DEVICE,
                                max_new_tokens=32, batch_size=8, seed=0)
        nll = metrics.reference_nll(model, tokenizer, prompts, out.texts, device=DEVICE)
        valid = ~nll.isnan()
        rows.append({
            "method": "B0" if r == 0.0 else "B1", "r": r, "feature_id": fid,
            "ppl": float(nll[valid].mean().exp()),
            "dist_2": metrics.distinct_n(out.texts, 2),
            "repetition_4": metrics.repetition_rate(out.texts, 4),
            "sample": out.texts[0],  # first prompt's continuation, for the eyeball check below
        })
    print(f"r={r} done")

import pandas as pd
df = pd.DataFrame(rows)
df["ppl_ratio"] = df["ppl"] / b0_ppl

print("\n--- aggregate (mean AND median")
print(df.groupby(["method", "r"])["ppl_ratio"].agg(["mean", "median", "max"]).round(2))

In [ ]:
for r in [0.5, 1.0]:
    sub = df[df.r == r].sort_values("ppl_ratio", ascending=False)
    print(f"\n--- r={r} ---")
    print(sub[["feature_id", "ppl_ratio", "dist_2"]].to_string(index=False))

In [ ]:
import json
descr = json.loads(open("../steering-final/results/vector_extraction.json").read()) \
    if False else None

for fid in [67922, 60695]:
    v = directions[FEATURE_IDS.index(fid)]
    print(f"\n=== feature {fid} ===")
    for r in [0.0, 0.2, 0.4, 0.6, 0.8, 1]:
        iv = interventions.NoSteering() if r == 0.0 else interventions.AdditiveSteering(v, r, scale)
        out = generate.generate(model, tokenizer, prompts[:1], iv, layer=LAYER, device=DEVICE,
                                max_new_tokens=40, seed=0)
        print(f"  r={r:>4}: {out.texts[0]!r}")

In [ ]:
R_GRID = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

io.run_or_load("step1_gate_b0_b1",
                {"model": "gpt2", "layer": LAYER, "r_grid": R_GRID,
                 "feature_ids": FEATURE_IDS, "n_prompts": len(prompts),
                 "max_new_tokens": 32, "seed": 0, "version": 2},
                lambda: df, force=True)

## Шаг 2. Создание выборки признаков для steering'а

In [13]:
import glob
import pandas as pd

pq = glob.glob(str(Path.home() / ".cache/huggingface/hub/datasets--NeelNanda--pile-10k"
                    "/snapshots/*/data/*.parquet"))[0]
corpus_texts = pd.read_parquet(pq)["text"].head(3000).tolist()
print(f"{len(corpus_texts)} documents for the frequency-band scan")

3000 documents for the frequency-band scan


In [14]:
center = spaces.should_center("gpt2")

stats = io.run_or_load(
    "feature_stats_gpt2_l6", {"model": "gpt2", "layer": LAYER, "n_docs": len(corpus_texts),
                               "max_length": 64, "center": center, "version": 1},
    lambda: vectors.compute_feature_stats(model, tokenizer, sae, layer=LAYER, texts=corpus_texts,
                                          batch_size=16, max_length=64, center=center),
    where="artifacts",
)
print(f"mean_l0={float(stats['mean_l0']):.2f}  (k={sae.cfg.k})  n_tokens={int(stats['n_tokens'])}")

cached  feature_stats_gpt2_l6.pt  (136ac70cbc3c1875)
mean_l0=32.00  (k=32)  n_tokens=186699


In [15]:
lo, hi = vectors.band_from_mean(stats, lo=0.41, hi=20.5)
in_band = vectors.frequency_band(stats, lo, hi)
print(f"band [{lo:.2e}, {hi:.2e}]  ->  {in_band.numel()} in-band candidates")

N_CANDIDATES = 1024
seed_gen = torch.Generator().manual_seed(0)
if in_band.numel() > N_CANDIDATES:
    idx = torch.randperm(in_band.numel(), generator=seed_gen)[:N_CANDIDATES]
    candidates = sorted(in_band[idx].tolist())
else:
    candidates = sorted(in_band.tolist())
print(f"{len(candidates)} candidates selected for probing")

ret = io.run_or_load("probe_candidates_gpt2",
                {"freq_lo": lo, "freq_hi": hi, "n_candidates": len(candidates), "seed": 0,
                 "n_in_band": int(in_band.numel()), "version": 1},
                lambda: {"candidates": candidates})

band [1.00e-04, 5.00e-03]  ->  41461 in-band candidates
1024 candidates selected for probing
cached  probe_candidates_gpt2.json  (98738db2b517fb71)


In [16]:
from steering import vectors

probe = io.run_or_load(
    "steerability_probe_gpt2",
    {"model": "gpt2", "layer": LAYER, "n_candidates": len(candidates),
     "scale": scale, "n_prompts": len(prompts), "max_new_tokens": 32, "seed": 0,
     "r_values": list(vectors.DEFAULT_PROBE_R), "version": 1},
    lambda: vectors.probe_steerability(model, tokenizer, sae, layer=LAYER,
                                       feature_ids=candidates, prompts=prompts,
                                       scale=scale, device=DEVICE, center=center,
                                       max_new_tokens=32, seed=0),
)

n_usable = sum(1 for r in probe.values() if r["usable"])
print(f"{len(probe)} probed -> {n_usable} usable (peak gain > {vectors.RESPONSIVE_THRESHOLD})")

steerability:   0%|          | 0/1024 [00:00<?, ?it/s]

computed steerability_probe_gpt2.json  (b749b3e1364c30a9)
1024 probed -> 159 usable (peak gain > 0.02)


In [17]:
usable_ids = sorted(f for f, r in probe.items() if r["usable"])
print(f"fetching descriptions for {len(usable_ids)} usable features")

descriptions = describe_features_cached = vectors.describe_features(
    sae.cfg.metadata.neuronpedia_id, usable_ids, cache_dir=io.ARTIFACTS / "neuronpedia_cache"
)
n_semantic = sum(1 for d in descriptions.values() if not d["token_level"])
print(f"{len(descriptions)} described -> {n_semantic} semantic (non-token-level)")

fetching descriptions for 159 usable features


neuronpedia:   0%|          | 0/159 [00:00<?, ?it/s]

159 described -> 131 semantic (non-token-level)


In [21]:
from steering import generate, interventions, metrics

survivors = sorted(f for f in usable_ids if descriptions.get(f) and not descriptions[f]["token_level"])
print(f"{len(survivors)} candidates entering replication (usable + described + semantic)")

directions_survivors = vectors.steering_directions(sae, survivors)

def replicate(seed=1):
    clean = generate.generate(model, tokenizer, prompts, interventions.NoSteering(),
                              layer=LAYER, device=DEVICE, max_new_tokens=32, seed=seed)
    out = {}
    for fid, v in zip(survivors, directions_survivors):
        base = metrics.sae_concept_score(model, tokenizer, sae, fid, LAYER, prompts,
                                         clean.texts, device=DEVICE, center=center)
        r_peak = probe[fid]["r_at_peak"]
        iv = interventions.AdditiveSteering(v, r=r_peak, scale=scale)
        steered = generate.generate(model, tokenizer, prompts, iv, layer=LAYER, device=DEVICE,
                                    max_new_tokens=32, seed=seed)
        score = metrics.sae_concept_score(model, tokenizer, sae, fid, LAYER, prompts,
                                          steered.texts, device=DEVICE, center=center)
        out[fid] = score["mean_act"] - base["mean_act"]
    return out

replication = io.run_or_load(
    "steerability_replication_gpt2",
    {"model": "gpt2", "layer": LAYER, "n_survivors": len(survivors), "seed": 1,
     "n_prompts": len(prompts), "version": 1},
    lambda: replicate(seed=1),
)

n_reproduced = sum(1 for f in survivors if replication.get(f, 0.0) > vectors.RESPONSIVE_THRESHOLD)
print(f"{len(survivors)} candidates -> {n_reproduced} reproduced under seed=1")

131 candidates entering replication (usable + described + semantic)
computed steerability_replication_gpt2.json  (680dc09614d6ad05)
131 candidates -> 105 reproduced under seed=1


In [23]:
split = vectors.select_and_freeze_split(
    stats, io.RESULTS / "feature_splits_gpt2.json",
    freq_min=lo, freq_max=hi, n_dev=35, n_test=70, seed=0,
    model="gpt2", source="gpt2_sae_blocks.6.hook_resid_post",
    steerability=probe, descriptions=descriptions, replication=replication,
)

print(f"DEV {len(split.dev)}, TEST {len(split.test)}, TRAIN pool {len(split.train_pool())}")
print(f"fingerprint {split.fingerprint()}")
print()
for k, v in split.selection.items():
    print(f"  {k}: {v}")

DEV 35, TEST 70, TRAIN pool 130967
fingerprint ce65a9496b253ceb

  method: steerability_then_description_then_replication
  freq_min: 0.00010009658231865614
  freq_max: 0.005004829115932807
  n_in_band: 41461
  n_probed: 1024
  n_usable: 159
  n_dropped_no_description: 0
  n_dropped_token_level: 28
  n_dropped_failed_replication: 26
  n_selectable: 105
  n_corpus_tokens: 186699


## Шаг 3. LLM-as-a-judge и первый тест

In [24]:
from steering import judge

split = vectors.load_split(io.RESULTS / "feature_splits_gpt2.json")
dev_directions = vectors.steering_directions(sae, split.dev)
print(f"{len(split.dev)} DEV concepts, split fingerprint {split.fingerprint()}")

35 DEV concepts, split fingerprint ce65a9496b253ceb


In [26]:
from tqdm.auto import tqdm

R_GRID = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
METHODS = {"B1": interventions.AdditiveSteering, "B2": interventions.NormMatchedSteering}

summary_rows, detail_rows = [], []

def record(method, r, fid, texts, nll):
    valid = ~nll.isnan()
    ppl = float(nll[valid].mean().exp()) if valid.any() else float("nan")
    score = metrics.sae_concept_score(model, tokenizer, sae, fid, LAYER, prompts, texts,
                                      device=DEVICE, center=center)
    summary_rows.append({"method": method, "r": r, "feature_id": fid, "ppl": ppl,
                         "dist_2": metrics.distinct_n(texts, 2),
                         "repetition_4": metrics.repetition_rate(texts, 4),
                         "concept_mean_act": score["mean_act"],
                         "concept_fire_rate": score["fire_rate"]})
    for i, (p, c, n) in enumerate(zip(prompts, texts, nll.tolist())):
        detail_rows.append({"method": method, "r": r, "feature_id": fid, "prompt_idx": i,
                           "prompt": p, "continuation": c, "nll": n})

b0 = generate.generate(model, tokenizer, prompts, interventions.NoSteering(), layer=LAYER,
                       device=DEVICE, max_new_tokens=32, batch_size=16, seed=0)
b0_nll = metrics.reference_nll(model, tokenizer, prompts, b0.texts, device=DEVICE)

for fid in tqdm(split.dev, desc="B0 concept scores"):
    record("B0", 0.0, fid, b0.texts, b0_nll)

for method_name, cls in METHODS.items():
    pairs = list(zip(split.dev, dev_directions))
    for fid, v in tqdm(pairs, desc=method_name):
        for r in R_GRID[1:]:
            iv = cls(v, r=r, scale=scale)
            out = generate.generate(model, tokenizer, prompts, iv, layer=LAYER, device=DEVICE,
                                    max_new_tokens=32, batch_size=16, seed=0)
            nll = metrics.reference_nll(model, tokenizer, prompts, out.texts, device=DEVICE)
            record(method_name, r, fid, out.texts, nll)

import pandas as pd

summary_config = {"model": "gpt2", "layer": LAYER, "split_fingerprint": split.fingerprint(),
                   "r_grid": R_GRID, "methods": [*METHODS, "B0"], "n_prompts": len(prompts),
                   "max_new_tokens": 32, "seed": 0, "version": 1}
summary_df = io.run_or_load("dev_sweep_summary_gpt2", summary_config, lambda: pd.DataFrame(summary_rows))
detail_df = io.run_or_load("dev_sweep_detail_gpt2", summary_config, lambda: pd.DataFrame(detail_rows),
                           where="artifacts")

B0 concept scores:   0%|          | 0/35 [00:00<?, ?it/s]

B1:   0%|          | 0/35 [00:00<?, ?it/s]

B2:   0%|          | 0/35 [00:00<?, ?it/s]

computed dev_sweep_summary_gpt2.csv  (9ec43e53ffb6195d)
computed dev_sweep_detail_gpt2.csv  (9ec43e53ffb6195d)


In [27]:
print(summary_df.groupby(["method", "r"])[
    ["ppl", "dist_2", "repetition_4", "concept_mean_act", "concept_fire_rate"]
].mean().round(4))

                 ppl  dist_2  repetition_4  concept_mean_act  \
method r                                                       
B0     0.0   50.6073  0.9646        0.0000            0.0013   
B1     0.2   63.7740  0.9633        0.0000            0.0150   
       0.4  106.4735  0.9537        0.0005            0.1028   
       0.6  250.6969  0.9349        0.0034            0.1808   
       0.8  486.5466  0.9108        0.0047            0.2324   
       1.0  734.8544  0.8817        0.0059            0.2745   
B2     0.2   63.2620  0.9633        0.0000            0.0148   
       0.4  104.0688  0.9564        0.0006            0.1035   
       0.6  262.7484  0.9348        0.0029            0.1582   
       0.8  571.3637  0.9090        0.0039            0.1927   
       1.0  915.2021  0.8781        0.0045            0.2005   

            concept_fire_rate  
method r                       
B0     0.0             0.0010  
B1     0.2             0.0093  
       0.4             0.0453  
       

Наблюдение: B2 проигрывает B1 по обеим осям при r≥0.6 и разрыв растет

In [28]:
rng_seed = 0
calibration_df = pd.concat(
    [g.sample(n=min(3, len(g)), random_state=rng_seed) for _, g in detail_df.groupby(["method", "r"])],
    ignore_index=True,
)
print(f"{len(calibration_df)} examples sampled for calibration")

j = judge.OllamaJudge(cache_dir=io.ARTIFACTS / "judge_cache")
calibration_df["judge_coherence"] = j.score_many(
    calibration_df["prompt"].tolist(), calibration_df["continuation"].tolist(),
    rubric="coherence", progress=True,
)
print(j.stats())

from scipy import stats as scipy_stats

valid = calibration_df["nll"].notna() & calibration_df["judge_coherence"].notna()
pear_r, pear_p = scipy_stats.pearsonr(calibration_df.loc[valid, "nll"], calibration_df.loc[valid, "judge_coherence"])
spear_r, spear_p = scipy_stats.spearmanr(calibration_df.loc[valid, "nll"], calibration_df.loc[valid, "judge_coherence"])
print(f"pearson r={pear_r:.3f} (p={pear_p:.4f})   spearman r={spear_r:.3f} (p={spear_p:.4f})")
# expect NEGATIVE correlation: higher NLL (less predictable text) should mean lower judge coherence

io.run_or_load("nll_judge_calibration_gpt2",
               {"model": "gpt2", "n_examples": len(calibration_df), "seed": rng_seed,
                "rubric": "coherence", "judge_model": j.model, "version": 1},
               lambda: calibration_df)

33 examples sampled for calibration


judge:coherence:   0%|          | 0/33 [00:00<?, ?it/s]

{'calls': 29, 'cache_hits': 4}
pearson r=-0.619 (p=0.0001)   spearman r=-0.766 (p=0.0000)
computed nll_judge_calibration_gpt2.csv  (296dd22a5c6b2b0e)


,method,r,feature_id,prompt_idx,prompt,continuation,nll,judge_coherence
0,B0,0.0,23651,25,The book describes how,its chapters describe physics impossible to e...,3.705442,50.0
1,B0,0.0,105157,29,"Under the bright lights,","waiting where he thought one or two boys, who...",4.398938,40.0
2,B0,0.0,105157,17,"After a few minutes,","in the doorway that led out of the building, ...",3.690382,85.0
3,B1,0.2,23651,25,The book describes how,its subjects were reared by a single caretake...,3.980356,85.0
4,B1,0.2,105157,29,"Under the bright lights,","waiting where he thought one or two boys, who...",4.596228,40.0
5,B1,0.2,105157,17,"After a few minutes,",in the doorway that led to the part that led ...,3.910162,40.0
6,B1,0.4,23651,25,The book describes how,its subjects were reared and then exposed to ...,4.509564,40.0
7,B1,0.4,105157,29,"Under the bright lights,","waiting where he thought one was as well, and...",4.909147,30.0
8,B1,0.4,105157,17,"After a few minutes,","in the doorway that led to Istvan, you could ...",3.804325,60.0
9,B1,0.6,23651,25,The book describes how,its mother's re-brain environment had abundan...,6.073830,85.0
